Merged Part 2 and Part 6 into unified Cross-Domain Queries section with vertical/horizontal traversal examples
*Co-authored with CoCo*

# Composable Semantic Views

This notebook demonstrates how **composable semantic views** enable cross-domain analytics 
through Snowflake's Cortex Analyst API. By composing semantic views via `IMPORTS`, 
organizations can build layered semantic models where:

- **Shared dimensions** are defined once and reused everywhere
- **Domain teams** own their local metrics while inheriting shared context
- **Executive dashboards** transitively access all layers without duplication
- **Cortex Analyst** generates correct cross-domain SQL automatically

### Prerequisites
Run `composable_sv_demo.sql` first to create the `DEMO_DB.GAMING` schema with all tables and semantic views.

## Architecture: 3-Layer Import Hierarchy

```
┌─────────────────────────────────────────────────────────────────────┐
│ Layer 3: SV_EXECUTIVE                                               │
│   imports SV_ENGAGEMENT → transitive to Layer 1                     │
│   Adds: M_SESSIONS_PER_PLAYER (new metric on imported entity)       │
└─────────────────────────┬───────────────────────────────────────────┘
                          │ IMPORTS
┌─────────────────────────┴───────────────────────────────────────────┐
│ Layer 2: Domain Semantic Views                                       │
│                                                                      │
│  SV_ENGAGEMENT          SV_STORE             SV_SUPPORT             │
│  ├─ imports L1          ├─ imports L1         ├─ imports SV_PLAYERS  │
│  └─ GAME_SESSIONS       ├─ PURCHASES          └─ SUPPORT_TICKETS   │
│                         └─ SUBSCRIPTIONS                            │
└──────────┬─────────────────────┬────────────────────┬───────────────┘
           │                     │                    │
┌──────────┴─────────────────────┴────────────────────┴───────────────┐
│ Layer 1: Shared Dimensions                                           │
│                                                                      │
│  SV_PLAYERS              SV_CONSOLE_CATALOG                         │
│  (15 players)            (5 consoles, 10 games)                     │
└─────────────────────────────────────────────────────────────────────┘
```

**Key insight**: `SV_EXECUTIVE` only directly imports `SV_ENGAGEMENT`, but transitively 
gains access to `SV_PLAYERS` and `SV_CONSOLE_CATALOG` — three levels deep.

In [ ]:
# Setup: imports and Snowpark session
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql('USE SCHEMA DEMO_DB.GAMING').collect()
session.sql('USE WAREHOUSE COMPUTE_WH').collect()
print('Connected to DEMO_DB.GAMING')

---
## Part 1: Exploring the Semantic View Hierarchy

Let's examine what semantic views exist and how they're connected through imports.

In [ ]:
# Show all semantic views and their import dependencies
df_svs = session.sql("SHOW SEMANTIC VIEWS IN SCHEMA DEMO_DB.GAMING").to_pandas()
df_svs.columns = df_svs.columns.str.strip('"').str.lower()
df_svs[['name', 'comment', 'imports']]

In [ ]:
# Visualize the import dependency graph
import ast

# Build edges from the imports column
edges = []
for _, row in df_svs.iterrows():
    sv_name = row['name']
    imports_str = row['imports']
    if imports_str and imports_str != '[]':
        imports_list = json.loads(imports_str)
        for imp in imports_list:
            source = imp.split('.')[-1]  # Extract SV name from FQN
            edges.append((source, sv_name))

# Create a Sankey-style flow diagram showing import relationships
nodes = list(df_svs['name'])
node_idx = {name: i for i, name in enumerate(nodes)}

# Assign layers for positioning
layers = {
    'SV_PLAYERS': 1, 'SV_CONSOLE_CATALOG': 1,
    'SV_ENGAGEMENT': 2, 'SV_STORE': 2, 'SV_SUPPORT': 2,
    'SV_EXECUTIVE': 3
}
colors = {1: '#2196F3', 2: '#4CAF50', 3: '#FF9800'}

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15, thickness=20,
        label=nodes,
        color=[colors.get(layers.get(n, 1), '#999') for n in nodes]
    ),
    link=dict(
        source=[node_idx[e[0]] for e in edges],
        target=[node_idx[e[1]] for e in edges],
        value=[1] * len(edges),
        color='rgba(100,100,100,0.3)'
    )
)])
fig.update_layout(
    title='Semantic View Import Dependencies',
    font_size=12, height=400
)
fig.show()

In [ ]:
# What does SV_EXECUTIVE see? (DESCRIBE shows imports + shadow tables + local metrics)
df_desc = session.sql('DESCRIBE SEMANTIC VIEW SV_EXECUTIVE').to_pandas()
df_desc

---
## Part 2: Cross-Domain Queries

Composable semantic views enable two styles of cross-domain querying:

- **Vertical traversal** — A top-level view (e.g. `SV_EXECUTIVE`) reaches *down* through
  multiple import layers to access dimensions and metrics defined several levels deep.
- **Horizontal sharing** — A domain-level view (e.g. `SV_STORE`, `SV_SUPPORT`) imports
  *sibling* base-layer views to combine its local metrics with shared dimensions.

Both styles use the same `SEMANTIC_VIEW()` table function — the composition graph is
resolved automatically regardless of traversal direction.

In [ ]:
# Helper: Query a semantic view using the SEMANTIC_VIEW() table function
def query_semantic_view(semantic_view_fqn: str, dimensions: list, metrics: list,
                       filters: str = None, order_by: str = None, limit: int = None,
                       verbose: bool = True):
    """
    Query a semantic view using the SEMANTIC_VIEW() SQL table function.
    Supports cross-domain queries via composable imports.
    Returns (sql, results_df).
    """
    dims_clause = ', '.join(dimensions) if dimensions else ''
    metrics_clause = ', '.join(metrics) if metrics else ''

    parts = [f"SELECT * FROM SEMANTIC_VIEW(\n        {semantic_view_fqn}"]
    if dims_clause:
        parts.append(f"        DIMENSIONS {dims_clause}")
    if metrics_clause:
        parts.append(f"        METRICS {metrics_clause}")
    parts.append("    )")

    sql = '\n'.join(parts)

    if filters:
        sql += f"\n    WHERE {filters}"
    if order_by:
        sql += f"\n    ORDER BY {order_by}"
    if limit:
        sql += f"\n    LIMIT {limit}"

    if verbose:
        print(f"SQL:\n{sql}\n")

    df = session.sql(sql).to_pandas()
    return sql, df

print('SEMANTIC_VIEW() query helper ready.')

### Vertical Traversal

These queries enter at `SV_EXECUTIVE` (Layer 3) and reach down through the import chain
to access dimensions defined in `SV_PLAYERS` or `SV_CONSOLE_CATALOG` (Layer 1):

```
SV_EXECUTIVE  →  imports SV_ENGAGEMENT  →  imports SV_PLAYERS / SV_CONSOLE_CATALOG
   (Layer 3)          (Layer 2)                     (Layer 1)
```

In [ ]:
# VERTICAL: SV_EXECUTIVE → SV_ENGAGEMENT → SV_PLAYERS
# Sessions per player by region (D_REGION is 2 layers deep)
sql, df = query_semantic_view(
    'DEMO_DB.GAMING.SV_EXECUTIVE',
    dimensions=['PLAYERS.D_REGION'],
    metrics=['GAME_SESSIONS.M_SESSIONS_PER_PLAYER', 'GAME_SESSIONS.M_TOTAL_SESSIONS'],
    order_by='M_TOTAL_SESSIONS DESC'
)
df

In [ ]:
# Visualize sessions by region
if df is not None and not df.empty:
    fig = px.bar(
        df,
        x='D_REGION',
        y='M_TOTAL_SESSIONS',
        color='D_REGION',
        title='Total Sessions by Player Region (via SV_EXECUTIVE → 3 layers deep)',
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    fig.update_layout(showlegend=False, xaxis_title='Region', yaxis_title='Sessions')
    fig.show()

### Horizontal Sharing

Domain-level views (`SV_STORE`, `SV_SUPPORT`) import shared base-layer views as
siblings — combining their own local metrics with shared dimensions from `SV_PLAYERS`.
No vertical depth, just peer-to-peer sharing:

```
SV_STORE  ─┐
            ├──→  imports SV_PLAYERS (shared base layer)
SV_SUPPORT ─┘
```

In [ ]:
# HORIZONTAL: SV_STORE imports SV_PLAYERS (sibling base layer)
# Revenue by region and purchase type (D_REGION from shared player dimension)
sql, df_revenue = query_semantic_view(
    'DEMO_DB.GAMING.SV_STORE',
    dimensions=['PLAYERS.D_REGION', 'PURCHASES.D_PURCHASE_TYPE'],
    metrics=['PURCHASES.M_TOTAL_REVENUE'],
    order_by='M_TOTAL_REVENUE DESC'
)
df_revenue

In [ ]:
# Revenue breakdown chart
if df_revenue is not None and not df_revenue.empty:
    fig = px.bar(
        df_revenue,
        x='D_REGION',
        y='M_TOTAL_REVENUE',
        color='D_PURCHASE_TYPE',
        barmode='group',
        title='Revenue by Region & Purchase Type (SV_STORE with imported PLAYERS dimension)',
        color_discrete_sequence=px.colors.qualitative.Pastel
    )
    fig.update_layout(xaxis_title='Region', yaxis_title='Revenue ($)')
    fig.show()

Another vertical traversal — this time reaching into a **different branch** of the import tree
(`SV_CONSOLE_CATALOG` instead of `SV_PLAYERS`):

```
SV_EXECUTIVE → SV_ENGAGEMENT → SV_CONSOLE_CATALOG → GAMES.D_GENRE
```

The `SEMANTIC_VIEW()` function resolves the join path between `GAME_SESSIONS` and `GAMES`
automatically — you never specify how they connect.

In [ ]:
# VERTICAL: SV_EXECUTIVE → SV_ENGAGEMENT → SV_CONSOLE_CATALOG
# Play time by genre (D_GENRE is 2 layers deep, from a different branch)
sql, df_genre = query_semantic_view(
    'DEMO_DB.GAMING.SV_EXECUTIVE',
    dimensions=['GAMES.D_GENRE'],
    metrics=['GAME_SESSIONS.M_TOTAL_PLAY_MINUTES'],
    order_by='M_TOTAL_PLAY_MINUTES DESC'
)
df_genre

In [ ]:
# Horizontal bar chart of play time by genre
if df_genre is not None and not df_genre.empty:
    df_sorted = df_genre.sort_values('M_TOTAL_PLAY_MINUTES', ascending=True)
    fig = px.bar(
        df_sorted,
        x='M_TOTAL_PLAY_MINUTES',
        y='D_GENRE',
        orientation='h',
        title='Total Play Minutes by Game Genre (SV_EXECUTIVE → SV_ENGAGEMENT → SV_CONSOLE_CATALOG)',
        color='M_TOTAL_PLAY_MINUTES',
        color_continuous_scale='Viridis'
    )
    fig.update_layout(yaxis_title='Genre', xaxis_title='Total Minutes', showlegend=False)
    fig.show()

Another horizontal sharing example — `SV_SUPPORT` imports `SV_PLAYERS` to slice its
ticket metrics by player tier, a dimension it doesn't own locally:

In [ ]:
# HORIZONTAL: SV_SUPPORT imports SV_PLAYERS (sibling base layer)
# Ticket count and satisfaction by player tier (D_TIER from shared player dimension)
sql, df_support = query_semantic_view(
    'DEMO_DB.GAMING.SV_SUPPORT',
    dimensions=['PLAYERS.D_TIER'],
    metrics=['SUPPORT_TICKETS.M_TICKET_COUNT', 'SUPPORT_TICKETS.M_AVG_SATISFACTION'],
    order_by='M_TICKET_COUNT DESC'
)

if df_support is not None and not df_support.empty:
    fig = px.scatter(
        df_support,
        x='M_TICKET_COUNT',
        y='M_AVG_SATISFACTION',
        size='M_TICKET_COUNT',
        text='D_TIER',
        title='Support: Satisfaction vs Ticket Volume by Player Tier (horizontal sharing)',
        color_discrete_sequence=['#E91E63']
    )
    fig.update_traces(textposition='top center')
    fig.update_layout(height=400)
    fig.show()

---
## Part 3: The Power of Transitive Imports

When `SV_EXECUTIVE` imports `SV_ENGAGEMENT`, it transitively inherits everything from 
`SV_PLAYERS` and `SV_CONSOLE_CATALOG`. This means a single `SEMANTIC_VIEW()` query 
can combine:

- **Player dimensions** (from Layer 1: `SV_PLAYERS`)
- **Game dimensions** (from Layer 1: `SV_CONSOLE_CATALOG`)
- **Session metrics** (from Layer 2: `SV_ENGAGEMENT`)
- **Custom executive metrics** (from Layer 3: `SV_EXECUTIVE`)

All without knowing the physical table structure or join paths.

In [ ]:
# Direct query: mix dimensions from 3 different layers in one call
df_transitive = session.sql("""
    SELECT * FROM SEMANTIC_VIEW(
        SV_EXECUTIVE
        DIMENSIONS PLAYERS.D_TIER, GAMES.D_GENRE
        METRICS GAME_SESSIONS.M_TOTAL_PLAY_MINUTES, GAME_SESSIONS.M_SESSIONS_PER_PLAYER
    )
    ORDER BY M_TOTAL_PLAY_MINUTES DESC
    LIMIT 15
""").to_pandas()

print("Dimensions from Layer 1 (PLAYERS.D_TIER, GAMES.D_GENRE)")
print("Metrics from Layer 2 (M_TOTAL_PLAY_MINUTES) and Layer 3 (M_SESSIONS_PER_PLAYER)")
print()
df_transitive

In [ ]:
# Heatmap: Play minutes by tier and genre
if not df_transitive.empty:
    tier_order = ['Bronze', 'Silver', 'Gold', 'Platinum']
    pivot = df_transitive.pivot_table(
        index='D_TIER', columns='D_GENRE',
        values='M_TOTAL_PLAY_MINUTES', aggfunc='sum'
    ).fillna(0)
    pivot = pivot.reindex([t for t in tier_order if t in pivot.index])
    
    fig = px.imshow(
        pivot,
        labels=dict(x='Game Genre', y='Player Tier', color='Minutes'),
        title='Play Minutes Heatmap: Player Tier × Game Genre (3 layers combined)',
        color_continuous_scale='YlOrRd',
        text_auto='.0f'
    )
    fig.update_layout(height=400, yaxis=dict(autorange='reversed'))
    fig.show()

---
## Part 4: Late Binding — Changes Propagate Instantly

One of the most powerful features of composable semantic views is **late binding**: 
when you modify a source semantic view, all downstream composed views immediately 
reflect the change — no recreation needed.

Let's prove it by adding a new dimension to `SV_PLAYERS` (Layer 1) and immediately 
querying it through `SV_EXECUTIVE` (Layer 3).

In [ ]:
# Current dimensions available in SV_PLAYERS
session.sql("DESCRIBE SEMANTIC VIEW SV_PLAYERS").collect()
df_players_dims = session.sql("""
    SELECT "object_name", "property_value"
    FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
    WHERE "object_kind" = 'DIMENSION' AND "property" = 'EXPRESSION'
""").to_pandas()
df_players_dims.columns = df_players_dims.columns.str.strip('"').str.lower()
print('Current SV_PLAYERS dimensions:')
df_players_dims

In [ ]:
# Add a new dimension: D_SIGNUP_YEAR (derived from D_SIGNUP_DATE)
session.sql("""
    CREATE OR REPLACE SEMANTIC VIEW SV_PLAYERS
      TABLES (
        DEMO_DB.GAMING.PLAYERS PRIMARY KEY (PLAYER_ID)
      )
      DIMENSIONS (
        PLAYERS.PLAYER_ID AS PLAYERS.PLAYER_ID COMMENT='Player ID dimension',
        PLAYERS.D_GAMER_TAG AS PLAYERS.GAMER_TAG COMMENT='Gamer tag dimension',
        PLAYERS.D_REGION AS PLAYERS.REGION COMMENT='Player region',
        PLAYERS.D_TIER AS PLAYERS.TIER COMMENT='Player loyalty tier',
        PLAYERS.D_SIGNUP_DATE AS PLAYERS.SIGNUP_DATE COMMENT='Player signup date',
        PLAYERS.D_EMAIL_DOMAIN AS SPLIT_PART(PLAYERS.EMAIL, '@', 2) COMMENT='Email domain',
        PLAYERS.D_SIGNUP_YEAR AS YEAR(PLAYERS.SIGNUP_DATE) COMMENT='Year player signed up — ADDED VIA LATE BINDING'
      )
      METRICS (
        PLAYERS.M_PLAYER_COUNT AS COUNT(DISTINCT PLAYERS.PLAYER_ID) COMMENT='Total unique players'
      )
      COMMENT='Shared player dimension — base layer for all domain SVs'
""").collect()

print('✓ Added D_SIGNUP_YEAR to SV_PLAYERS (Layer 1)')
print('  SV_ENGAGEMENT (Layer 2) and SV_EXECUTIVE (Layer 3) were NOT touched.')

Confirm Late Binding on the Player Sign Up Year dimension is accessible

In [ ]:
# Immediately query the NEW dimension through SV_EXECUTIVE (3 layers deep!)
df_late = session.sql("""
    SELECT * FROM SEMANTIC_VIEW(
        SV_EXECUTIVE
        DIMENSIONS PLAYERS.D_SIGNUP_YEAR
        METRICS GAME_SESSIONS.M_TOTAL_SESSIONS, GAME_SESSIONS.M_AVG_SESSION_MINUTES
    )
    ORDER BY D_SIGNUP_YEAR
""").to_pandas()

print('Late binding confirmed! D_SIGNUP_YEAR propagated through 3 layers instantly.')
print()
df_late

In [ ]:
# Visualize: sessions and duration by signup year cohort
if not df_late.empty:
    fig = px.bar(
        df_late,
        x='D_SIGNUP_YEAR',
        y='M_TOTAL_SESSIONS',
        title='Sessions by Player Signup Year (new dimension via late binding)',
        text='M_TOTAL_SESSIONS',
        color_discrete_sequence=['#FF9800']
    )
    fig.update_layout(xaxis_title='Signup Year', yaxis_title='Total Sessions')
    fig.update_traces(textposition='outside')
    fig.show()

---
## Part 5: Introspection & Governance

Composable semantic views provide rich metadata for governance and discovery. 
Let's explore the tools available to understand the composition structure.

1. **Composability is declarative** — the dependency between semantic views is part of the schema definition itself, not hidden in code or config files.

2. **Reproducibility** — you can GET_DDL, version it in git, and recreate the exact same composition hierarchy in another environment.

3. **Transparency** — anyone with access can see exactly which views feed into SV_EXECUTIVE just by inspecting its DDL, without needing to trace through application code.

In [ ]:
# GET_DDL shows the IMPORTS clause — proving composition is first-class DDL
ddl = session.sql("SELECT GET_DDL('SEMANTIC_VIEW', 'DEMO_DB.GAMING.SV_EXECUTIVE')").collect()[0][0]
print('DDL for SV_EXECUTIVE:')
print('=' * 60)
print(ddl)

In [ ]:
# YAML shows the full transitive model including imported_semantic_models
import yaml

yaml_str = session.sql(
    "SELECT SYSTEM$READ_YAML_FROM_SEMANTIC_VIEW('DEMO_DB.GAMING.SV_EXECUTIVE')"
).collect()[0][0]

model = yaml.safe_load(yaml_str)

# Show the top-level structure
print(f"Semantic View: {model['name']}")
print(f"Description: {model['description']}")
print(f"\nDirect imports: {len(model.get('imports', []))}")
for imp in model.get('imports', []):
    print(f"  → {imp['database_name']}.{imp['schema_name']}.{imp['name']}")

print(f"\nTransitively resolved models: {len(model.get('imported_semantic_models', []))}")
for m in model.get('imported_semantic_models', []):
    fqn = m['fully_qualified_name']
    tables = m.get('tables', [])
    local_tables = [t['name'] for t in tables if not t.get('is_imported')]
    imported_tables = [t['name'] for t in tables if t.get('is_imported')]
    print(f"  {fqn['name']}:")
    if local_tables:
        print(f"    Local tables: {local_tables}")
    if imported_tables:
        print(f"    Imported (shadow): {imported_tables}")

**EntitiesPerView** cell shows, for each semantic view, how many table references it has and whether those tables are owned locally or pulled in via IMPORTS. A view with many imported entities is heavily composed from other views; one with only local entities is a standalone base layer.

In [ ]:
# Visualize: which entities are local vs imported across all SVs
sv_data = []
for sv_name in ['SV_PLAYERS', 'SV_CONSOLE_CATALOG', 'SV_ENGAGEMENT', 'SV_STORE', 'SV_SUPPORT', 'SV_EXECUTIVE']:
    try:
        y = session.sql(f"SELECT SYSTEM$READ_YAML_FROM_SEMANTIC_VIEW('DEMO_DB.GAMING.{sv_name}')").collect()[0][0]
        m = yaml.safe_load(y)
        for t in m.get('tables', []):
            sv_data.append({
                'Semantic View': sv_name,
                'Entity': t['name'],
                'Type': 'Imported (shadow)' if t.get('is_imported') else 'Local',
                'Metrics': len(t.get('metrics', [])),
                'Dimensions': len(t.get('dimensions', []))
            })
    except:
        pass

df_entities = pd.DataFrame(sv_data)

fig = px.bar(
    df_entities.groupby(['Semantic View', 'Type']).size().reset_index(name='Count'),
    x='Semantic View', y='Count', color='Type',
    barmode='stack',
    title='Local vs Imported Entities per Semantic View',
    color_discrete_map={'Local': '#4CAF50', 'Imported (shadow)': '#FF9800'}
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

---
## Cleanup: Remove Late Binding Demo Dimension

Restore `SV_PLAYERS` to its original state by removing the `D_SIGNUP_YEAR` dimension 
that was added during the late binding demo.

In [ ]:
# Restore SV_PLAYERS to original state (remove D_SIGNUP_YEAR added in late binding demo)
session.sql("""
    CREATE OR REPLACE SEMANTIC VIEW SV_PLAYERS
      TABLES (
        DEMO_DB.GAMING.PLAYERS PRIMARY KEY (PLAYER_ID)
      )
      DIMENSIONS (
        PLAYERS.PLAYER_ID AS PLAYERS.PLAYER_ID COMMENT='Player ID dimension',
        PLAYERS.D_GAMER_TAG AS PLAYERS.GAMER_TAG COMMENT='Gamer tag dimension',
        PLAYERS.D_REGION AS PLAYERS.REGION COMMENT='Player region',
        PLAYERS.D_TIER AS PLAYERS.TIER COMMENT='Player loyalty tier',
        PLAYERS.D_SIGNUP_DATE AS PLAYERS.SIGNUP_DATE COMMENT='Player signup date',
        PLAYERS.D_EMAIL_DOMAIN AS SPLIT_PART(PLAYERS.EMAIL, '@', 2) COMMENT='Email domain'
      )
      METRICS (
        PLAYERS.M_PLAYER_COUNT AS COUNT(DISTINCT PLAYERS.PLAYER_ID) COMMENT='Total unique players'
      )
      COMMENT='Shared player dimension — base layer for all domain SVs'
""").collect()

print('✓ SV_PLAYERS restored — D_SIGNUP_YEAR removed.')
print('  All downstream views (SV_ENGAGEMENT, SV_EXECUTIVE, etc.) automatically reflect the change.')

---
## Key Takeaways

### Why Composable Semantic Views?

| Benefit | Without Composition | With Composition |
|---------|--------------------|-----------------|
| **Shared definitions** | Copy-paste dimensions across SVs | Define once, import everywhere |
| **Cross-domain queries** | Manual JOINs across models | Automatic via transitive imports |
| **Change propagation** | Update N views when schema changes | Update source, all consumers reflect instantly |
| **Governance** | Hard to trace lineage | DESCRIBE/YAML shows full import chain |
| **Cortex Analyst** | Limited to single-domain answers | Understands full composed model |

### Design Rules

1. **No diamond imports** — Two paths to the same ancestor are not allowed
2. **PK dimensions must match column names** — For relationship ref-keys to work
3. **Imported entities are read-only calculations** — Can't reference physical columns
4. **Late binding is instant** — `CREATE OR REPLACE` on source propagates everywhere
5. **Privilege isolation works** — `REFERENCES + SELECT` on composed view is sufficient

### Architecture Pattern

```
Layer 1: Shared dimensions (players, products, geography)
    ↓ IMPORTS
Layer 2: Domain metrics (engagement, revenue, support)
    ↓ IMPORTS  
Layer 3: Executive/cross-domain (composed KPIs, analyst-ready)
```

This pattern gives domain teams autonomy over their metrics while maintaining 
a single source of truth for shared dimensions.